# TDC-KV Colab Quickstart

Use this notebook for small Google Colab tests before running expensive 7B/8B experiments.

Recommended Colab runtime for this notebook:

- Runtime type: Python 3
- Hardware accelerator: GPU
- For trace tests: T4 is enough
- For tiny HF smoke tests: T4/L4 is enough
- For real 7B/8B tests: prefer L4/A100

Important: if you clone from GitHub, Colab only sees committed/pushed code. If your local working tree has changes that are not pushed, either push them first or use the zip-upload path below.

In [ ]:
# 1. Check GPU and Python environment
import os, platform, subprocess, sys

print('Python:', sys.version)
print('Platform:', platform.platform())

try:
    import torch
    print('Torch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except Exception as exc:
    print('Torch check failed:', repr(exc))

subprocess.run(['nvidia-smi'], check=False)

## 2. Get The Code

Default path: clone from GitHub. If you need local unpushed changes, set `USE_GITHUB = False`, run the cell, and upload a `.zip` of the project folder.

In [ ]:
# 2. Get the project code
from pathlib import Path
import shutil

USE_GITHUB = True
REPO_URL = 'https://github.com/JayGor-13/Tier-based-KV-cache-with-Dependency-Aware-Chunk-Scoring.git'
PROJECT_DIR = Path('/content/Tier-based-KV-cache-with-Dependency-Aware-Chunk-Scoring')

if USE_GITHUB:
    shutil.rmtree(PROJECT_DIR, ignore_errors=True)
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(PROJECT_DIR)], check=True)
else:
    from google.colab import files
    import zipfile
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No zip uploaded.')
    zip_name = next(iter(uploaded.keys()))
    shutil.rmtree(PROJECT_DIR, ignore_errors=True)
    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_name) as zf:
        zf.extractall(PROJECT_DIR)
    nested = [p for p in PROJECT_DIR.iterdir() if p.is_dir() and (p / 'src').exists()]
    if nested:
        PROJECT_DIR = nested[0]

os.chdir(PROJECT_DIR)
print('Project dir:', Path.cwd())
print('Files:', sorted(p.name for p in Path.cwd().iterdir())[:20])

In [ ]:
# 3. Install lightweight dependencies
# Do not reinstall torch on Colab unless you have a specific CUDA reason.
packages = [
    'pytest',
    'numpy',
    'scipy',
    'matplotlib',
    'accelerate',
    'datasets',
    'transformers',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)

os.environ['PYTHONPATH'] = str(Path.cwd()) + os.pathsep + os.environ.get('PYTHONPATH', '')
print('PYTHONPATH starts with:', os.environ['PYTHONPATH'].split(os.pathsep)[0])

In [ ]:
# 4. Import sanity check
import src.core
import src.baselines
from benchmarks.hf_runner import normalize_methods

print('imports ok')
print('methods:', normalize_methods(['fullkv', 'streamingllm', 'h2o', 'snapkv', 'chunkkv', 'tdc-kv']))

In [ ]:
# 5. Run fast unit tests
subprocess.run([
    sys.executable, '-m', 'pytest',
    'tests/test_chunker.py',
    'tests/test_scorer.py',
    'tests/test_masker.py',
    'tests/test_evictor.py',
    'tests/test_baselines.py',
    '-q'
], check=True)

In [ ]:
# 6. Run trace-driven smoke tests. These are cheap and should work even on T4.
Path('outputs').mkdir(exist_ok=True)

subprocess.run([
    sys.executable, 'scripts/run_main_results.py',
    '--trace-path', 'data/sample_trace.jsonl',
    '--recent-window', '4',
    '--output', 'outputs/colab_main_smoke.json'
], check=True)

subprocess.run([
    sys.executable, 'scripts/run_baselines.py',
    '--trace-path', 'data/sample_trace.jsonl',
    '--methods', 'streamingllm,chunkkv,snapkv,h2o',
    '--recent-window', '4',
    '--output', 'outputs/colab_baselines_smoke.json'
], check=True)

subprocess.run([
    sys.executable, 'scripts/run_ablations.py',
    '--trace-path', 'data/sample_trace.jsonl',
    '--theta-grid', '0.2,0.3',
    '--recent-window-grid', '4,8',
    '--output', 'outputs/colab_ablations_smoke.json'
], check=True)

In [ ]:
# 7. Inspect smoke outputs
import json

for path in [
    'outputs/colab_main_smoke.json',
    'outputs/colab_baselines_smoke.json',
    'outputs/colab_ablations_smoke.json',
]:
    print('\n===', path, '===')
    with open(path, 'r', encoding='utf-8') as f:
        payload = json.load(f)
    if 'summary' in payload:
        print(json.dumps(payload['summary'], indent=2)[:1200])
    elif 'cache_summary' in payload:
        print(json.dumps(payload['cache_summary'], indent=2))
    else:
        print(json.dumps(payload, indent=2)[:1200])

## 8. Optional Tiny HuggingFace Smoke Test

This checks model loading, attention extraction, chunk scoring, and multi-method HF-grid output. It uses `sshleifer/tiny-gpt2` and `max_new_tokens=0`, so it is a pipeline smoke test, not a quality result.

In [ ]:
# 8. Optional tiny HF smoke test
tiny_records = [
    {'id': 'toy_0', 'prompt': 'Question: What is 2 plus 2?\nAnswer:', 'answer': '4'},
]
tiny_path = Path('data/colab_tiny_qa.jsonl')
with tiny_path.open('w', encoding='utf-8') as f:
    for rec in tiny_records:
        f.write(json.dumps(rec) + '\n')

dataset_spec = 'name=tiny,source=data/colab_tiny_qa.jsonl,prompt_field=prompt,answer_field=answer,id_field=id'
subprocess.run([
    sys.executable, 'scripts/run_hf_grid.py',
    '--models', 'sshleifer/tiny-gpt2',
    '--datasets', dataset_spec,
    '--budget-ratios', '0.5',
    '--thetas', '0.3',
    '--recent-windows', '4',
    '--alphas', '0.6',
    '--methods', 'fullkv,streamingllm,h2o,snapkv,chunkkv,tdc_kv',
    '--max-samples', '1',
    '--max-length', '128',
    '--max-new-tokens', '0',
    '--device', 'auto',
    '--dtype', 'auto',
    '--output', 'outputs/colab_tiny_hf_grid.json',
], check=True)

with open('outputs/colab_tiny_hf_grid.json', 'r', encoding='utf-8') as f:
    hf_payload = json.load(f)
print(json.dumps(hf_payload['summary'], indent=2)[:2000])

## 9. Next Small Real Test

After the tiny smoke test works, try a small real model on 2 to 5 examples. Use this only on GPU.

Suggested first model:

- `Qwen/Qwen2.5-0.5B-Instruct`

Then, if it works:

- `Qwen/Qwen2.5-1.5B-Instruct`
- `Qwen/Qwen2.5-3B-Instruct`

Do not start with 7B/8B until the full pipeline passes on small models.